<small>📌 <b>먼저 내 노트로 저장하기</b></small><br><small>이 노트는 내 구글 드라이브의 『Colab Notebooks』 폴더(자동 사본으로 만들었으면 『배화 Colab 실습』 폴더)에 저장돼요.</small><br><small>사이트의 「▶ 내 Colab 노트 만들기」로 열었으면 사본·공유가 이미 돼 있어요 — 아래 ①~④ 는 건너뛰어도 돼요.</small><br><br><small>① 맨 위 <b>파일 → Drive에 사본 저장</b> (안 하면 쓴 게 사라져요)</small><br><img src="https://baehwa-ai-2026.web.app/aisocial/img/colab_step_copy.jpg" width="300"><br><br><small>② 사본이 열리면 오른쪽 위 <b>공유</b> 단추를 눌러요.</small><br><img src="https://baehwa-ai-2026.web.app/aisocial/img/colab_step_share.jpg" width="300"><br><br><small>③ 일반 액세스에서 <b>「링크가 있는 모든 사용자」</b>를 고른 다음 <b>링크 복사</b>.</small><br><img src="https://baehwa-ai-2026.web.app/aisocial/img/colab_step_anyone.jpg" width="300"><br><br><small>④ 사이트로 돌아와 <b>『📓 내 Colab 실습 노트』</b> 칸에 붙여넣고 <b>『보관』</b>을 눌러요 — <b>『✅ 보관됨』</b>이 뜨면 선생님께 간 거예요.</small><br><img src="https://baehwa-ai-2026.web.app/aisocial/img/colab_step_paste.jpg" width="300">

<small>☁ <b>저장 확인</b> — 맨 위 노트 이름 옆에 <b>체크된 구름</b>이 보이면 내 드라이브에 <b>실시간으로 저장</b>되고 있어요. (구름이 없고 「Drive로 복사」가 보이면 아직 내 노트가 아니에요 — 파일 → Drive에 사본 저장)</small><br>

![체크된 구름 = 저장 잘 되는 중](https://raw.githubusercontent.com/SoojiNoh/baehwa-colab/main/nb/img/colab_saved.png)

# 🧠 영양성분 세 가지의 열량 — 딥러닝(신경망) 실습

관련 특성을 추가했을 때 설명력이 변함을 확인한다. 예상: 예상 R² 0.95~1.0 부근으로 높을 수 있다.

- **데이터 출처**: Kaggle · mcdonalds/nutrition-facts — https://www.kaggle.com/datasets/mcdonalds/nutrition-facts
- **라이선스**: Unknown
- **이 카드의 질문**: 관련 특성을 추가했을 때 설명력이 변함을 확인한다.

속성(열) 뜻은 수행평가 사이트 카드의 「속성(변수) 설명」 표를 보세요.

In [ ]:
# 0. 준비 — 그대로 실행해요 (한글 그래프 글꼴)
!pip -q install koreanize-matplotlib
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
import numpy as np
import keras
keras.utils.set_random_seed(0)   # 매번 같은 결과가 나오게


## 1. 데이터 불러오기
📖 **오픈북** — 「#20 과일 채소 딥러닝」 숙제00 `pd.read_csv(url)`

In [ ]:
df = pd.______('https://baehwa-ai-2026.web.app/dataproject/csv/kaggle/menu.csv')      # 힌트: CSV 를 읽는 함수
print(df.shape)
df.head()

## 2. 맞힐 열(y)과 넣을 열(X) 고르기 · 0~1 로 맞추기
`y_col` 을 다른 열로 바꿔 봐도 돼요. 숫자 종류가 적거나 글자면 **분류**, 숫자면 **회귀**로 저절로 정해져요.

In [ ]:
y_col = 'Calories'          # 맞힐 열
drop_cols = ['Calories from Fat']   # 넣지 않을 열(번호·정답과 겹치는 열)
df = df.dropna(subset=[y_col])
Xdf = df.drop(columns=[y_col] + drop_cols, errors='ignore').select_dtypes('number')
if Xdf.shape[1] == 0:
    Xdf = pd.DataFrame({'순서': range(len(df))})   # 숫자 열이 없으면 「몇 번째 줄인가」를 넣어요
Xdf = Xdf.fillna(Xdf.mean())
X = ((Xdf - Xdf.min()) / (Xdf.max() - Xdf.min()).replace(0, 1)).values   # 정규화 0~1
is_cls = df[y_col].dtype == object or df[y_col].dtype == bool or df[y_col].nunique() <= 25
if is_cls:
    names, y = np.unique(df[y_col].astype(str), return_inverse=True)
else:
    y = df[y_col].astype(float).values
    y_mean, y_std = y.mean(), y.std() or 1
    y = (y - y_mean) / y_std
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)
print('분류' if is_cls else '회귀', '· 넣는 열', X.shape[1], '개 · 학습용', len(X_train), '줄')

## 3. 신경망 설계하기
입력층 → 은닉층(32) → 은닉층(16) → 출력층. 분류면 종류 수만큼(softmax), 회귀면 숫자 1개.

📖 **오픈북** — 「#20 과일 채소 딥러닝」 숙제01 `Sequential()` · `Dense()`

In [ ]:
from keras.models import Sequential
from keras.layers import Dense, Input
model = ______()      # 힌트: 층을 차례로 쌓는 모델
model.add(Input(shape=(X.shape[1],)))
model.add(Dense(32, activation='relu'))
model.add(Dense(16, activation='relu'))
model.add(Dense(len(names), activation='softmax') if is_cls else Dense(1))

## 4. 학습 방법 정하기 (compile) → 학습 (fit) → 평가 (evaluate)
📖 **오픈북** — 「#20 과일 채소 딥러닝」 숙제02·03·04

In [ ]:
model.______(loss='sparse_categorical_crossentropy' if is_cls else 'mse', optimizer='adam', metrics=['accuracy'] if is_cls else ['mae'])      # 힌트: 학습 방법 정하기
model.______(X_train, y_train, epochs=30, verbose=0)      # 힌트: 학습시키기
print('시험 [손실, 정확도 또는 평균 오차]', model.evaluate(X_test, y_test, verbose=0))

## 5. 예측해 보기 (predict)
📖 **오픈북** — 「MNIST 딥러닝」 숙제09 `model.predict`

In [ ]:
p = model.______(X_test[:5], verbose=0)      # 힌트: 예측하기
if is_cls:
    print('예측', names[np.argmax(p, axis=1)], '\n실제', names[y_test[:5]])
else:
    print('예측', (p[:, 0] * y_std + y_mean).round(1), '\n실제', (y_test[:5] * y_std + y_mean).round(1))

## 6. 결과 읽기 (선택)
✍️ **(선택 — 필수 아니에요)** 답은 수행평가 사이트 칸에 써요.
- 정확도(또는 평균 오차)가 기대만큼 나왔나요? → 「1-2」
- 신경망이 잘 못 맞힌다면 데이터의 어떤 점 때문일까요? → 「1-3. 한계」